# Run GMEL For One Config

Use this notebook to train one GMEL configuration, save the best checkpoint, and save the corresponding source/destination embeddings. Edit the config cell, then run the notebook top to bottom.

In [1]:
from pathlib import Path
import logging
import os
import warnings

import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

# Make relative repo paths work whether Jupyter starts in repo root or in code/.
if Path("code/train.py").exists():
    os.chdir("code")

import utils
from model import MyModel

warnings.filterwarnings(
    "ignore",
    message="Feature table contains NaN. 0 is used to fill these NaNs"
)

Path("models").mkdir(exist_ok=True)
Path("embeddings").mkdir(exist_ok=True)
Path("log").mkdir(exist_ok=True)
Path("runs").mkdir(exist_ok=True)

print("cwd:", Path.cwd())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

cwd: c:\Users\qshah\Documents\Spring 2026\GMEL\code
torch: 2.3.0
cuda available: True


In [5]:
YEAR = 2015
NUM_HIDDEN_LAYERS = 1
EMBEDDING_SIZE = 128
MULTITASK_WEIGHTS = (0.5, 0.25, 0.25)

# DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DEVICE = "cpu"

MAX_EPOCHS = 120
MINI_BATCH_SIZE = 1e3
NEGATIVE_SAMPLING_RATE = 0
LR = 1e-5
GRAD_NORM = 1.0
REG_PARAM = 0
EVALUATE_EVERY = 5

SKIP_IF_OUTPUTS_EXIST = False

args = {
    "year": YEAR,
    "device": DEVICE,
    "embedding_size": EMBEDDING_SIZE,
    "num_hidden_layers": NUM_HIDDEN_LAYERS,
    "reg_param": REG_PARAM,
    "multitask_weights": MULTITASK_WEIGHTS,
    "max_epochs": MAX_EPOCHS,
    "mini_batch_size": MINI_BATCH_SIZE,
    "negative_sampling_rate": NEGATIVE_SAMPLING_RATE,
    "lr": LR,
    "grad_norm": GRAD_NORM,
    "evaluate_every": EVALUATE_EVERY,
}

model_state_file = Path(
    f"./models/model_state_layers{NUM_HIDDEN_LAYERS}_emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.pth"
)
embedding_file = Path(
    f"./embeddings/censustract_embeddings_year{YEAR}_layers{NUM_HIDDEN_LAYERS}_emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.npz"
)

print("Using device:", DEVICE)
print("Checkpoint:", model_state_file)
print("Embeddings:", embedding_file)
print("Config:", args)

Using device: cpu
Checkpoint: models\model_state_layers1_emb128_multitask(0.5, 0.25, 0.25).pth
Embeddings: embeddings\censustract_embeddings_year2015_layers1_emb128_multitask(0.5, 0.25, 0.25).npz
Config: {'year': 2015, 'device': 'cpu', 'embedding_size': 128, 'num_hidden_layers': 1, 'reg_param': 0, 'multitask_weights': (0.5, 0.25, 0.25), 'max_epochs': 120, 'mini_batch_size': 1000.0, 'negative_sampling_rate': 0, 'lr': 1e-05, 'grad_norm': 1.0, 'evaluate_every': 5}


In [6]:
def train_single_gmel(args):
    device = torch.device(args["device"])
    run_name = "#layers{}_emb{}_multitask{}".format(
        args["num_hidden_layers"],
        args["embedding_size"],
        args["multitask_weights"],
    )

    writer = SummaryWriter(comment=run_name)
    logger = logging.getLogger("single_gmel" + run_name)
    logger.handlers.clear()
    logger.setLevel(logging.DEBUG)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    file_handler = logging.FileHandler("log/training_log.log")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(logging.StreamHandler())

    torch.manual_seed(2019)
    np.random.seed(2019)

    data = utils.load_dataset(year=args["year"])
    train_data = data["train"]
    valid_data = data["valid"]
    test_data = data["test"]
    train_inflow = data["train_inflow"]
    train_outflow = data["train_outflow"]
    node_feats = data["node_feats"]
    ct_adj = data["ct_adjacency_withweight"]
    num_nodes = data["num_nodes"]

    train_data = torch.from_numpy(train_data)
    trip_od_train = train_data[:, :2].long().to(device)
    trip_volume_train = train_data[:, -1].float().to(device)
    trip_od_valid = torch.from_numpy(valid_data[:, :2]).long().to(device)
    trip_volume_valid = torch.from_numpy(valid_data[:, -1]).float().to(device)
    trip_od_test = torch.from_numpy(test_data[:, :2]).long().to(device)
    trip_volume_test = torch.from_numpy(test_data[:, -1]).float().to(device)

    train_inflow = torch.from_numpy(train_inflow).view(-1, 1).float().to(device)
    train_outflow = torch.from_numpy(train_outflow).view(-1, 1).float().to(device)

    # Build on CPU first, then move the entire DGL graph to the target device.
    g = utils.build_graph_from_matrix(ct_adj, node_feats.astype(np.float32), device="cpu").to(device)

    model = MyModel(
        g,
        num_nodes,
        in_dim=node_feats.shape[1],
        h_dim=args["embedding_size"],
        num_hidden_layers=args["num_hidden_layers"],
        dropout=0,
        device=device,
        reg_param=args["reg_param"],
    ).to(device)

    model_state_file = "./models/model_state_layers{}_emb{}_multitask{}.pth".format(
        args["num_hidden_layers"],
        args["embedding_size"],
        args["multitask_weights"],
    )
    best_rmse = 1e6

    optimizer = torch.optim.Adam(model.parameters(), lr=args["lr"])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 100, gamma=0.1)

    for epoch in range(args["max_epochs"]):
        model.train()
        mini_batch_gen = utils.mini_batch_gen(
            train_data,
            mini_batch_size=int(args["mini_batch_size"]),
            num_nodes=num_nodes,
            negative_sampling_rate=args["negative_sampling_rate"],
        )

        for mini_batch in mini_batch_gen:
            optimizer.zero_grad()
            trip_od = mini_batch[:, :2].long().to(device)
            scaled_trip_volume = utils.scale(mini_batch[:, -1].float()).to(device)
            loss = model.get_loss(
                trip_od,
                scaled_trip_volume,
                train_inflow,
                train_outflow,
                g,
                multitask_weights=args["multitask_weights"],
            )
            writer.add_scalar("mini_loss", loss.item(), global_step=epoch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args["grad_norm"])
            optimizer.step()

        scheduler.step()

        model.eval()
        with torch.no_grad():
            train_loss = model.get_loss(
                trip_od_train,
                utils.scale(trip_volume_train),
                train_inflow,
                train_outflow,
                g,
                multitask_weights=args["multitask_weights"],
            )

        rmse, mae, mape, cpc, cpl = utils.evaluate(model, g, trip_od_train, trip_volume_train)
        writer.add_scalar("overall-loss", train_loss.item(), epoch)
        writer.add_scalar("RMSE", rmse, epoch)
        writer.add_scalar("MAE", mae, epoch)
        writer.add_scalar("MAPE", mape, epoch)
        writer.add_scalar("CPC", cpc, epoch)
        writer.add_scalar("CPL", cpl, epoch)

        if epoch % args["evaluate_every"] == 0:
            with torch.no_grad():
                valid_loss = model.get_loss(
                    trip_od_valid,
                    utils.scale(trip_volume_valid),
                    train_inflow,
                    train_outflow,
                    g,
                    multitask_weights=args["multitask_weights"],
                )
            rmse, mae, mape, cpc, cpl = utils.evaluate(model, g, trip_od_valid, trip_volume_valid)
            logger.info(
                "Epoch %04d | valid loss %.4f | RMSE %.4f | MAE %.4f | MAPE %.4f | CPC %.4f | CPL %.4f",
                epoch,
                valid_loss.item(),
                rmse,
                mae,
                mape,
                cpc,
                cpl,
            )

            if rmse < best_rmse:
                best_rmse = rmse
                torch.save(
                    {
                        "state_dict": model.state_dict(),
                        "epoch": epoch,
                        "rmse": rmse,
                        "mae": mae,
                        "mape": mape,
                        "cpc": cpc,
                        "cpl": cpl,
                    },
                    model_state_file,
                )
                src_embedding = model(g).detach().cpu().numpy()
                dst_embedding = model.forward2(g).detach().cpu().numpy()
                emb_fp = "./embeddings/censustract_embeddings_year{}_layers{}_emb{}_multitask{}.npz".format(
                    args["year"],
                    args["num_hidden_layers"],
                    args["embedding_size"],
                    args["multitask_weights"],
                )
                np.savez(emb_fp, src_embedding, dst_embedding)
                logger.info("Saved new best checkpoint and embeddings from epoch %s", epoch)

    writer.close()
    return model_state_file

In [ ]:
if SKIP_IF_OUTPUTS_EXIST and model_state_file.exists() and embedding_file.exists():
    print("Skipping training because outputs already exist.")
    print("Checkpoint:", model_state_file)
    print("Embeddings:", embedding_file)
else:
    saved_checkpoint = train_single_gmel(args)
    print("Training finished. Best checkpoint:", saved_checkpoint)
    print("Embeddings file:", embedding_file)

In [ ]:
assert model_state_file.exists(), f"Missing checkpoint: {model_state_file}"
assert embedding_file.exists(), f"Missing embeddings: {embedding_file}"

checkpoint = torch.load(model_state_file, map_location="cpu")
embeddings = np.load(embedding_file)

print("checkpoint epoch:", checkpoint.get("epoch"))
print("checkpoint rmse:", checkpoint.get("rmse"))
print("src embedding shape:", embeddings["arr_0"].shape)
print("dst embedding shape:", embeddings["arr_1"].shape)